# When should patients be admitted, and where should they recover?

**An independent decision study by Abdulaziz Aldoseri.**

A hospital planner has a fixed cohort with surgical disciplines, admission windows, expected surgery durations and recovery stays. The model chooses one admission day and one eligible ward for every new patient. Surgery occurs on admission day; a patient stays in that ward for the entire prescribed stay. Beds are hard limits. Surgery time beyond the given discipline/day allocation is recorded as additional capacity required, not assumed operational capacity.

Compare two priorities: **admission delay first, then excess surgery minutes**, and **excess surgery minutes first, then admission delay**. These are two lexicographic endpoints, not a full Pareto frontier. A deterministic deadline-first insertion/backtracking schedule provides a comparator only when it places every patient. A separate strict-calendar diagnostic asks whether all patients fit without any excess surgery time.

**Source:** Pieter Smet, [Data for “Generating balanced workload allocations in hospitals”](https://data.mendeley.com/datasets/3mv4rtxtfs/1), Mendeley Data, version 1, 13 April 2023, DOI 10.17632/3mv4rtxtfs.1, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). These are generated benchmark instances informed by one Belgian hospital, not observed patient records. Independent modifications preserve selected native data and patient requirements while replacing the paper's workload-balancing objectives with a timing/capacity comparison. No source-author or hospital endorsement is implied.

This deterministic whole-day model is not clinical guidance, a staff roster, an operating-room timetable, a forecast of emergency arrivals or evidence of realized hospital improvement.


## 1. Verify the complete reproduction package

The website's single **Open in Google Colab** link opens a delivery edition that automatically retrieves and validates the complete pinned package. No separate data upload or Drive access is required. Hosted Colab execution is distinct from local verification; see the release record for the environment actually checked.

For local use, open this notebook inside the extracted package with Python 3.12 or later and the exact packages in `requirements.txt`. Local setup makes no network request and installs nothing. The embedded standard-library helper validates the package manifest and all declared code/data hashes before study imports. It permits edits to the working notebook while code, requirements and evidence stay hash-checked.

The package contains the 20 selected native instances, prepared inputs, the frozen protocol/model, all 40 cases including unavailable/infeasible states, tests and source attribution. The original paper's PDF is not redistributed. Checksums establish consistency with the pinned package, not publisher identity or scientific correctness.


In [ ]:
"""Standard-library ZIP validation for the healthcare admission reproduction notebook.

Checksums detect corruption and manifest inconsistency, not publisher identity.
Obtain the archive from the study's own download link and compare its published
archive checksum when available. No archive code is executed by this helper.
"""
from __future__ import annotations

from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import re
import stat
from zipfile import ZipFile

MAX_ARCHIVE_BYTES = 100 * 1024 * 1024
MAX_TOTAL_BYTES = 256 * 1024 * 1024
MAX_FILE_BYTES = 64 * 1024 * 1024
MAX_FILES = 2000
MANIFEST_NAME = "package_manifest.json"


def safe_name(name: str) -> str:
    if not isinstance(name, str) or not name or "\\" in name or ":" in name or "\x00" in name:
        raise ValueError("Invalid archive path")
    path = PurePosixPath(name)
    if path.is_absolute() or any(part in {"", ".", ".."} for part in name.split("/")):
        raise ValueError("Archive path must be relative without traversal")
    if any(part.casefold() in {"private", "private_audit", ".git", ".env", ".deps"} for part in path.parts):
        raise ValueError("Private or repository-internal material is not allowed in the public package")
    if path.suffix.casefold() in {".xlsx", ".xls"} or "journeydataextract" in path.name.casefold():
        raise ValueError("The public package must not contain raw source workbooks or journey CSVs")
    return path.as_posix()


def manifest_entries(raw: bytes) -> dict:
    if len(raw) > 1024 * 1024:
        raise ValueError("Manifest exceeds size limit")
    value = json.loads(raw.decode("utf-8"))
    if not isinstance(value, dict) or not isinstance(value.get("files"), list):
        raise ValueError("Manifest must contain a files array")
    if not 1 <= len(value["files"]) <= MAX_FILES:
        raise ValueError("Invalid manifest file count")
    records, folded = {}, set()
    for item in value["files"]:
        if not isinstance(item, dict):
            raise ValueError("Invalid manifest record")
        name = safe_name(item.get("path"))
        size, digest = item.get("bytes"), item.get("sha256")
        if name == MANIFEST_NAME or name.casefold() in folded:
            raise ValueError("Duplicate, case-colliding or self-referencing manifest path")
        if isinstance(size, bool) or not isinstance(size, int) or not 0 <= size <= MAX_FILE_BYTES:
            raise ValueError("Invalid manifest size")
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Invalid SHA-256 digest")
        records[name] = {"bytes": size, "sha256": digest}
        folded.add(name.casefold())
    if sum(record["bytes"] for record in records.values()) > MAX_TOTAL_BYTES:
        raise ValueError("Manifest total exceeds size limit")
    return records


def verify_package(directory: str | Path, allow_notebook_edits: bool = False) -> dict:
    """Verify declared files; optional working-notebook edits do not exempt code/data.

    Extraction always uses strict verification. At notebook runtime, Jupyter
    saves outputs and edited parameters into .ipynb files, so those working
    documents may differ while all executable modules/data/requirements remain
    hash-checked. Notebook paths still must exist and be regular bounded files.
    """
    root = Path(directory).resolve()
    manifest = root / MANIFEST_NAME
    if manifest.is_symlink() or not manifest.is_file():
        raise ValueError("Package manifest is missing or is a symlink")
    entries = manifest_entries(manifest.read_bytes())
    for name, item in entries.items():
        target = root / name
        if any(part.is_symlink() for part in [target, *target.parents] if part != root.parent):
            raise ValueError("Symlinks are not permitted in a package")
        if not target.resolve().is_relative_to(root) or not target.is_file():
            raise ValueError("Missing or unsafe package file")
        if allow_notebook_edits and target.suffix.casefold() == ".ipynb":
            if target.stat().st_size > MAX_FILE_BYTES:
                raise ValueError("Working notebook exceeds size limit")
            continue
        if target.stat().st_size != item["bytes"] or sha256(target.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("Package checksum or size mismatch: " + name)
    return entries


def safe_extract_package(archive: bytes | str | Path, destination: str | Path) -> Path:
    """Verify all members before writing to a new/empty destination directory."""
    if isinstance(archive, bytes):
        raw = archive
    else:
        path = Path(archive)
        if path.stat().st_size > MAX_ARCHIVE_BYTES:
            raise ValueError("Archive exceeds compressed size limit")
        raw = path.read_bytes()
    if len(raw) > MAX_ARCHIVE_BYTES:
        raise ValueError("Archive exceeds compressed size limit")
    output = Path(destination)
    if output.is_symlink():
        raise ValueError("Destination must not be a symlink")
    output = output.resolve()
    if output.exists() and (not output.is_dir() or any(output.iterdir())):
        raise ValueError("Choose a new or empty extraction directory")
    with ZipFile(BytesIO(raw)) as archive_zip:
        infos = archive_zip.infolist()
        if len(infos) > MAX_FILES + 1:
            raise ValueError("Archive has too many entries")
        names, folded, total = {}, set(), 0
        for info in infos:
            name = safe_name(info.filename.rstrip("/") if info.is_dir() else info.filename)
            if name.casefold() in folded:
                raise ValueError("Duplicate or case-colliding archive member")
            folded.add(name.casefold())
            kind = stat.S_IFMT(info.external_attr >> 16)
            if kind not in {0, stat.S_IFREG, stat.S_IFDIR} or (kind == stat.S_IFDIR and not info.is_dir()):
                raise ValueError("Archive contains a nonregular entry")
            if info.flag_bits & 1:
                raise ValueError("Encrypted archive members are not supported")
            if info.is_dir():
                continue
            if not 0 <= info.file_size <= MAX_FILE_BYTES:
                raise ValueError("Archive member exceeds size limit")
            total += info.file_size
            names[name] = info
        if total > MAX_TOTAL_BYTES:
            raise ValueError("Archive exceeds expanded size limit")
        file_names = {name.casefold() for name in names}
        prefixes = {}
        for name in names:
            parts = PurePosixPath(name).parts
            for end in range(1, len(parts) + 1):
                prefix = "/".join(parts[:end])
                folded_prefix = prefix.casefold()
                if folded_prefix in prefixes and prefixes[folded_prefix] != prefix:
                    raise ValueError("Inconsistent case in archive path components")
                prefixes[folded_prefix] = prefix
                if end < len(parts) and folded_prefix in file_names:
                    raise ValueError("Archive file conflicts with a required directory")
        if MANIFEST_NAME not in names:
            raise ValueError("Archive root must contain package_manifest.json")
        if names[MANIFEST_NAME].file_size > 1024 * 1024:
            raise ValueError("Manifest exceeds size limit")
        manifest = archive_zip.read(names[MANIFEST_NAME])
        entries = manifest_entries(manifest)
        if set(names) != set(entries) | {MANIFEST_NAME}:
            raise ValueError("Every archive file must appear exactly once in the manifest")
        verified = {MANIFEST_NAME: manifest}
        for name, item in entries.items():
            info = names[name]
            if info.file_size != item["bytes"]:
                raise ValueError("Member size disagrees with manifest")
            with archive_zip.open(info) as member:
                contents = member.read(MAX_FILE_BYTES + 1)
            if len(contents) != item["bytes"] or sha256(contents).hexdigest() != item["sha256"]:
                raise ValueError("Member checksum disagrees with manifest: " + name)
            verified[name] = contents
    # The manifest is validated before any data or executable source is written.
    output.mkdir(parents=True, exist_ok=True)
    for name, contents in verified.items():
        target = output / name
        if not target.resolve().is_relative_to(output):
            raise ValueError("Unsafe extraction target")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as handle:
            handle.write(contents)
    verify_package(output)
    return output


In [ ]:
import importlib.metadata
import json
import subprocess
import sys

PROJECT = Path.cwd().resolve()
if not (PROJECT / "package_manifest.json").is_file():
    raise ValueError("The reproduction package manifest is required before study imports. Open the notebook inside the extracted package or use the website's Colab link.")
verified_files = verify_package(PROJECT, allow_notebook_edits=True)
sys.path.insert(0, str(PROJECT))
# This tail is also retained by the automatic Colab delivery edition.
import importlib.metadata
import json
import subprocess
import sys
import tempfile
for requirement in (PROJECT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    if not requirement.strip() or requirement.lstrip().startswith("#"):
        continue
    name, version = requirement.strip().split("==")
    if importlib.metadata.version(name) != version:
        raise RuntimeError(f"Use the pinned requirement {name}=={version}; restart the runtime if another version is already loaded.")
    loaded = sys.modules.get(name)
    if loaded is not None and getattr(loaded, "__version__", version) != version:
        raise RuntimeError(f"{name} was already imported at another version. Restart the runtime and run setup again.")
import pandas as pd
from scheduling_model import verify_schedule
from prepare_data import parse_native
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)
protocol = json.loads((PROJECT / "PROTOCOL.json").read_text(encoding="utf-8"))
checked = subprocess.run([sys.executable, str(PROJECT / "pipeline.py"), "verify"], cwd=PROJECT, text=True, capture_output=True, check=True)
print(checked.stdout.strip())
print(f"Verified {len(verified_files)} package files, frozen model inputs and saved evaluation outputs.")


## 2. Understand the information available to the planner

The cohorts are the five smallest numeric native seeds, 0 through 4, each with all four original minor-specialty variants M=0,1,2,3. Every new patient, source ward, admission window and calendar day is retained. The two bed scenarios close zero or one bed **in each ward** throughout the seven-day census. The closure is a hypothetical scenario, not an observed disruption.

Native days 0 through 6 display as Day 1 through Day 7. They are not calendar dates or weekdays. Surgery durations and scheduled allocations are integer minutes. A patient admitted on day a with LOS L occupies days a through a+L-1; LOS=1 occupies the admission day. All allowable new-patient stays fit the native seven days.

Carryover is the source's fixed ward/day census, not individual preexisting patient records. Some inherited patients remain on the final day; their later discharge dates are unknown. No later occupancy is invented. Staff/workload fields, clinical urgency weights and unavailable procedure IDs are not added to the model. Native ward/specialty codes are retained without guessed expansions.


In [ ]:
input_rows = []
for seed in range(5):
    for variant in range(4):
        filename = f"s{seed}m{variant}"
        native = parse_native(PROJECT / "sources" / "original" / "instances" / (filename + ".dat"))
        prepared = json.loads((PROJECT / "data" / "instances" / (filename + ".json")).read_text(encoding="utf-8"))
        assert native == prepared, f"Native-to-prepared reconciliation changed: {filename}"
        assert all(p["latest"] + p["los"] <= native["days"] for p in native["patients"])
        input_rows.append({"Cohort": seed, "M": variant, "New patients": len(native["patients"]), "Fixed admission dates": sum(p["earliest"] == p["latest"] for p in native["patients"]), "Days": native["days"], "Wards": len(native["wards"])})
display(pd.DataFrame(input_rows))
print("All 20 prepared instances match their complete native source records under the documented mapping.")


## 3. Check every saved case, including failures

Each binary decision assigns a patient to one eligible ward on one allowed day. For every ward/day, fixed carryover plus all admitted patients whose stays include that day must be no greater than source beds minus closures. Surgical load is the sum of exact durations assigned to a discipline/day. Excess is recomputed as max(0, assigned minutes minus scheduled minutes).

Delay-first proves minimum total patient-days of delay, fixes that value exactly, then minimizes excess. Excess-first reverses the priorities. Both stages must prove optimality for a lexicographic optimum. Limits retain their actual status; no secondary solve can turn an unproven primary incumbent into an optimum.

The baseline sorts by deadline, earliest day and frozen source patient order, then tries earlier days and the major ward before minor wards in frozen source order. It uses bounded backtracking and stops at its first complete schedule. Hitting a search bound is baseline unavailability. The strict-calendar check uses identical patients and hard beds; it tests the original calendar without silently changing either primary schedule.

The following cell reconstructs every available schedule independently of solver slack and constraint matrices. Missing schedules remain missing. It reports all 40 cases and all three policies, including ties, limits and infeasibility.


In [ ]:
expected_ids = [f"s{s}m{m}b{b}" for s in range(5) for m in range(4) for b in (0, 1)]
case_files = {p.stem: p for p in (PROJECT / "outputs" / "cases").glob("*.json")}
assert set(case_files) == set(expected_ids), "Complete 40-case grid is required."
cases = {key: json.loads(case_files[key].read_text(encoding="utf-8")) for key in expected_ids}
summary_rows = []
verified_schedules = 0
missing_schedules = 0
for key, case in cases.items():
    assert case["id"] == key
    for policy in ("baseline", "delay_first", "overtime_first"):
        result = case["schedules"][policy]
        if result["assignments"] is None:
            assert all(result[field] is None for field in ("metrics", "occupancy", "or_minutes"))
            missing_schedules += 1
        else:
            reconstructed = verify_schedule(case["instance"], result["assignments"], case["parameters"]["closed_beds"])
            assert reconstructed["verification"]["complete"] and reconstructed["verification"]["feasible"]
            for field in ("metrics", "occupancy", "or_minutes"):
                assert reconstructed[field] == result[field], (key, policy, field)
            verified_schedules += 1
        metrics = result["metrics"] or {}
        summary_rows.append({"Case": key, "Policy": policy, "Status": result["status"], "Delay, patient-days": metrics.get("delay_days"), "Excess, minutes": metrics.get("overtime_minutes"), "Strict calendar": case["strict_calendar"]["status"]})
evidence_check = {"status": "PASS", "cases": len(cases), "policy_records": len(summary_rows), "complete_verified_schedules": verified_schedules, "missing_schedules_retained": missing_schedules}
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(pd.DataFrame(summary_rows))
print(evidence_check)


## 4. Inspect the default choice and its complete daily census

The default was fixed before evaluation: cohort 0, M=0, no bed closures, delay-first. Change only the case ID and policy below to inspect another evaluated record. This selects saved evidence; it does not interpolate or solve an arbitrary scenario. Patient IDs refer to generated benchmark entries only.

The ward table separates inherited occupancy from newly assigned patients. The surgery table compares assigned minutes with the given discipline/day allocation. If a calendar cell has no scheduled time, any load requires an additional session; it should not be interpreted as overtime in an existing session. The selected native cohorts have positive calendar allocations; zero-calendar behavior is also checked in synthetic model tests.


In [ ]:
CASE_ID = "s0m0b0"
POLICY = "delay_first"
if CASE_ID not in cases or POLICY not in ("baseline", "delay_first", "overtime_first"):
    raise ValueError("Choose an evaluated case and one of the three displayed policies.")
case = cases[CASE_ID]
selected = case["schedules"][POLICY]
print(CASE_ID, POLICY, selected["status"], selected["message"])
print("Strict calendar:", case["strict_calendar"]["status"], case["strict_calendar"]["message"])
if selected["assignments"] is None:
    print("No complete verified schedule is available; no previous plot or patient detail is retained.")
else:
    print(selected["metrics"])
    ward_rows = []
    for wi, ward in enumerate(case["instance"]["wards"]):
        for day in range(case["instance"]["days"]):
            total = selected["occupancy"][wi][day]
            inherited = ward["carryover"][day]
            capacity = ward["capacity"] - case["parameters"]["closed_beds"]
            ward_rows.append({"Ward": ward["id"], "Day": day + 1, "Inherited": inherited, "New": total - inherited, "Occupied": total, "Beds": capacity, "Free": capacity - total})
    surgery_rows = []
    for si, specialty in enumerate(case["instance"]["specialties"]):
        for day, capacity in enumerate(specialty["minutes"]):
            usage = selected["or_minutes"][si][day]
            surgery_rows.append({"Discipline": specialty["id"], "Day": day + 1, "Scheduled, min": capacity, "Assigned, min": usage, "Excess, min": max(0, usage - capacity), "Additional session, min": usage if capacity == 0 else 0})
    patients = {p["id"]: p for p in case["instance"]["patients"]}
    stays = [{"Patient": a["patient_id"], "Discipline": patients[a["patient_id"]]["specialty"], "Ward": a["ward_id"], "Admission day": a["day"] + 1, "Last occupied day": a["day"] + patients[a["patient_id"]]["los"], "Delay, days": a["day"] - patients[a["patient_id"]]["earliest"]} for a in selected["assignments"]]
    with pd.option_context("display.max_rows", None):
        display(pd.DataFrame(ward_rows))
        display(pd.DataFrame(surgery_rows))
        display(pd.DataFrame(stays))


## 5. Exercise the implementation checks

These synthetic fixtures challenge scheduling logic and source validation; they are not hospital evidence. They include exact assignment reconstruction, eligibility/windows, capacity and carryover, whole-stay boundaries, baseline limits/backtracking, competing objective priorities, solver-status handling and safe archive setup. Passing implementation checks does not establish clinical or operational suitability.


In [ ]:
test_run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT / "tests"), "-v"], cwd=PROJECT, text=True, capture_output=True, check=True)
print(test_run.stderr or test_run.stdout)
print("The actual packaged implementation and archive-setup tests passed.")


## 6. Optionally solve the full grid again

The package above already supplies and verifies complete frozen results. Set `RERUN_FULL_GRID = True` to run a fresh 40-case evaluation under the same frozen protocol and compare it with the original evidence. Solver stages have explicit time limits; this can take several minutes. A new temporary directory preserves all original files. Re-running the cell produces another new directory.

Equal objective pairs do not guarantee identical assignments: alternative optimal schedules are possible. The comparison records status, feasibility and objective differences and reports assignment differences without treating every alternative optimum as a defect. Time-limited incumbents can differ across machines. A successful local rerun is not proof of hosted Colab execution.


In [ ]:
RERUN_FULL_GRID = bool(globals().get("RERUN_FULL_GRID", False))  # Set to True for a fresh full-grid solve.
reproduction_check = None
if RERUN_FULL_GRID:
    RUN = Path(tempfile.mkdtemp(prefix="healthcare-reproduction-")) / "outputs"
    rerun = subprocess.run([sys.executable, str(PROJECT / "pipeline.py"), "reproduce", "--output-dir", str(RUN)], cwd=PROJECT, text=True, capture_output=True, check=True)
    print(rerun.stdout.strip())
    reproduction_check = json.loads((RUN / "reproduction_comparison.json").read_text(encoding="utf-8"))
    assert reproduction_check["status"] == "pass"
    assignment_differences = []
    for key, original in cases.items():
        current = json.loads((RUN / "cases" / (key + ".json")).read_text(encoding="utf-8"))
        for policy in ("baseline", "delay_first", "overtime_first"):
            if original["schedules"][policy]["assignment_sha256"] != current["schedules"][policy]["assignment_sha256"]:
                assignment_differences.append({"case": key, "policy": policy})
    reproduction_check["assignment_differences"] = assignment_differences
    print("Assignment differences (objective/status comparison above remains authoritative):", assignment_differences)
else:
    print("Fresh optimization was not requested. All saved cases were independently reconstructed and the packaged tests were executed above.")


## 7. Keep the conclusion within the evidence

Delay and surgical excess are separate quantities with separate units. There is no invented financial or clinical conversion between them. Extra eligible wards can offer more assignment choices while real staffing and care constraints remain unmodelled. Closing beds is an explicit sensitivity scenario. A strict-calendar failure describes this fixed generated cohort and calendar; it does not identify a real hospital's bottleneck or propose treatment changes.

Keep all 40 cases, unsuccessful comparators, solver limits and ties in summaries. This small predetermined selection is not a random sample or a population estimate. Cite the dataset creator, DOI, version and CC BY 4.0 licence alongside exported evidence, and identify the independent transformation/objective changes.

Original study code is licensed separately in `LICENSE`. The protocol, source manifest, model freeze, complete outputs and tests in this package provide the detailed audit trail. Publication and hosted execution are separate release events.
